# 02_pl_test - Converted from Synapse Pipeline: 01_pl_silver

This notebook replicates the Synapse pipeline logic in PySpark:
1. Logs pipeline start execution
2. Gets metadata from stored procedure
3. Iterates through notebooks to run
4. Handles parameters conditionally
5. Logs success/failure


In [0]:
# Import required libraries
from pyspark.sql import SparkSession
from datetime import datetime
import json

# Notebook parameters (widgets)
dbutils.widgets.text("Environment", "silver", "Environment")
environment = dbutils.widgets.get("Environment")

In [0]:
# Pipeline context variables
pipeline_name = "02_pl_test"  # Equivalent to @pipeline().Pipeline
pipeline_run_id = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get() + "_" + str(datetime.now().timestamp())
pipeline_workspace = dbutils.notebook.entry_point.getDbutils().notebook().getContext().tags().get("browserHostName").getOrElse("databricks")

print(f"Pipeline: {pipeline_name}")
print(f"Run ID: {pipeline_run_id}")
print(f"Workspace: {pipeline_workspace}")
print(f"Environment: {environment}")

In [0]:
# Step 1: Log Start Execution Pipeline
# Call stored procedure to log pipeline start
# NOTE: Update the database/catalog name based on your environment

try:
    # Example: CALL your_catalog.config.LogStartExec(pipeline_name, pipeline_run_id, pipeline_workspace)
    log_start_query = f"""
    SELECT 
        '{pipeline_name}' as PipelineName,
        '{pipeline_run_id}' as PipelineRunID,
        '{pipeline_workspace}' as PipelineWorkspace,
        current_timestamp() as StartTime,
        1 as ID_Log
    """
    
    # TODO: Replace with actual stored procedure call
    # log_start_result = spark.sql("CALL config.LogStartExec(:pipeline_name, :pipeline_run_id, :pipeline_workspace)")
    log_start_result = spark.sql(log_start_query)
    
    # Get the ID_Log
    id_log = log_start_result.first()["ID_Log"]
    print(f"✓ Logged pipeline start. ID_Log: {id_log}")
    
except Exception as e:
    print(f"⚠ Warning: Could not log pipeline start: {str(e)}")
    id_log = -1

In [0]:
# Step 2: Get Metadata
# Call stored procedure to get list of notebooks to run
# NOTE: Update the database/catalog name based on your environment

try:
    # TODO: Replace with actual stored procedure call
    # metadata_query = f"CALL config.GetMetadata('{environment}')"
    # metadata_df = spark.sql(metadata_query)
    
    # Example metadata structure (replace with actual stored procedure call)
    metadata_query = f"""
    SELECT 
        'NotebookName1' as Notebook,
        false as NeedParameters,
        'account1' as ADLSaccount,
        'container1' as containerName,
        'bronze' as sourceFolder,
        '{environment}' as targetFolder,
        '{environment}' as targetEnv,
        'entity1' as entityName,
        'parquet' as fileExtension
    """
    
    metadata_df = spark.sql(metadata_query)
    metadata_list = metadata_df.collect()
    
    print(f"✓ Retrieved metadata: {len(metadata_list)} notebook(s) to run")
    display(metadata_df)
    
except Exception as e:
    print(f"✗ Error getting metadata: {str(e)}")
    # Log failure
    if id_log != -1:
        try:
            # TODO: Call LogEndExec stored procedure with Failed status
            print(f"Logging failure for ID_Log: {id_log}")
        except:
            pass
    raise

In [0]:
# Step 3: ForEach - Iterate through notebooks
# Process notebooks sequentially (isSequential: true in pipeline)

notebook_results = []
failed_notebooks = []

for idx, item in enumerate(metadata_list, 1):
    notebook_name = item["Notebook"]
    need_parameters = bool(item["NeedParameters"])
    
    print(f"\n{'='*60}")
    print(f"Processing {idx}/{len(metadata_list)}: {notebook_name}")
    print(f"Need Parameters: {need_parameters}")
    print(f"{'='*60}")
    
    try:
        if need_parameters:
            # Run notebook WITH parameters
            notebook_params = {
                "ADLSaccount": item["ADLSaccount"],
                "containerName": item["containerName"],
                "sourceFolder": item["sourceFolder"],
                "targetFolder": item["targetFolder"],
                "targetEnv": item["targetEnv"],
                "entityName": item["entityName"],
                "fileExtension": item["fileExtension"]
            }
            
            print(f"Running with parameters: {json.dumps(notebook_params, indent=2)}")
            
            # Run the notebook (timeout: 12 hours = 43200 seconds)
            result = dbutils.notebook.run(
                notebook_name, 
                timeout_seconds=43200,
                arguments=notebook_params
            )
            
            print(f"✓ Notebook completed successfully")
            notebook_results.append({"notebook": notebook_name, "status": "success", "result": result})
            
        else:
            # Run notebook WITHOUT parameters
            print(f"Running without parameters")
            
            result = dbutils.notebook.run(
                notebook_name,
                timeout_seconds=43200
            )
            
            print(f"✓ Notebook completed successfully")
            notebook_results.append({"notebook": notebook_name, "status": "success", "result": result})
            
    except Exception as e:
        print(f"✗ Notebook failed: {str(e)}")
        failed_notebooks.append({"notebook": notebook_name, "error": str(e)})
        notebook_results.append({"notebook": notebook_name, "status": "failed", "error": str(e)})
        # Continue with next notebook or stop? Original pipeline doesn't have stopOnFailure
        # If you want to stop on first failure, uncomment the next line:
        # raise

print(f"\n{'='*60}")
print(f"Completed processing {len(metadata_list)} notebook(s)")
print(f"Successful: {len([r for r in notebook_results if r['status'] == 'success'])}")
print(f"Failed: {len(failed_notebooks)}")
print(f"{'='*60}")

In [0]:
# Step 4: Log End Execution
# Determine overall status and log

overall_status = "Succeeded" if len(failed_notebooks) == 0 else "Failed"

try:
    # TODO: Replace with actual stored procedure call
    # log_end_query = f"CALL config.LogEndExec({id_log}, '{overall_status}')"
    # spark.sql(log_end_query)
    
    print(f"\n✓ Pipeline execution logged with status: {overall_status}")
    print(f"ID_Log: {id_log}")
    
except Exception as e:
    print(f"⚠ Warning: Could not log pipeline end: {str(e)}")

# Return results
if overall_status == "Failed":
    print(f"\n⚠ Pipeline completed with failures:")
    for failed in failed_notebooks:
        print(f"  - {failed['notebook']}: {failed['error']}")
    dbutils.notebook.exit(json.dumps({"status": "Failed", "failed_notebooks": failed_notebooks}))
else:
    print(f"\n✓ Pipeline completed successfully")
    dbutils.notebook.exit(json.dumps({"status": "Succeeded", "notebooks_processed": len(metadata_list)}))